In [1]:
# import sleap
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, Flatten, Reshape, Dense
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from scipy.io import loadmat, savemat
import os
import h5py

2025-02-02 17:02:43.021349: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-02-02 17:02:43.036354: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1738544563.053999  347075 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1738544563.059173  347075 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-02 17:02:43.078067: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  2


## autoencoder class (example)

In [7]:
# source: https://github.com/MweinbergUmass/Masters_Thesis/blob/main/Pose_Reconstruction/Util/TC_Auto.py
class Autoencoder:
    
    def __init__(self, 
                 input_shape, 
                 bottleneck_size=22, 
                 activation='selu', 
                 conv_units=256, 
                 kernel_size=3, 
                 stride_size=1, 
                 batch_size=512, 
                 dropout_rate=0.1):
        
        self.input_shape = input_shape
        self.bottleneck_size = bottleneck_size
        self.activation = activation
        self.conv_units = conv_units
        self.kernel_size = kernel_size
        self.stride_size = stride_size
        self.batch_size = batch_size
        self.dropout_rate = dropout_rate
        self.model = self.build_model()

    def build_model(self):
        # Encoder
        inputs = Input(shape=self.input_shape)  # Input shape should be (sequence_length, num_features)
        x = Conv1D(self.conv_units, kernel_size=self.kernel_size, strides=self.stride_size, activation=self.activation, padding='same')(inputs)
        x = Flatten()(x)
        encoded = Dense(self.bottleneck_size, activation=self.activation)(x)
        # Decoder
        x = Dense(np.prod(self.input_shape), activation=self.activation)(encoded)
        x = Reshape(self.input_shape)(x)
        x = Conv1D(self.conv_units, kernel_size=self.kernel_size, strides=self.stride_size, activation=self.activation, padding='same')(x)
        decoded = Conv1D(self.input_shape[1], kernel_size=self.kernel_size, strides=self.stride_size, activation='linear', padding='same')(x)
        # Autoencoder
        autoencoder = Model(inputs, decoded)
        autoencoder.compile(optimizer='adam', loss='mse')
        autoencoder.summary()
        return autoencoder
    
    def train(self, x_train_masked, x_train, x_val_masked, x_val, epochs=100,ER_Patience=25, LR_patience=10):
        early_stopping = EarlyStopping(monitor='val_loss', patience=ER_Patience, restore_best_weights=True)
        reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=LR_patience, min_lr=1e-6)
        self.model.fit(x_train_masked, x_train, epochs=epochs, batch_size=self.batch_size, validation_data=(x_val_masked, x_val), callbacks=[early_stopping, reduce_lr])
        
    def predict(self, data):
        return self.model.predict(data)
    
    def evaluate(self, x_test, y_test):
        return self.model.evaluate(x_test, y_test)
    
    def save_model(self, save_path):
        self.model.save(save_path)

In [ ]:
# I want a function which simply reads in an h5 file, given a path, and a dataset name
def h5read(file_path, dataset_name):
    with h5py.File(file_path, 'r') as file:
        data = file[dataset_name][:]
    return data
def load_v73_mat_file(file_path, var_name='X_data_all'):
    with h5py.File(file_path, 'r') as file:
        X_data_all = file[var_name][:]
        X_data_all = X_data_all.T
    return X_data_all
def load_data(file_path):
    try:
        data = loadmat(file_path)
        train_dataX = data['train_dataX']
        train_dataY = data['train_dataY']
        test_dataX = data['test_dataX']
        test_dataY = data['test_dataY']
    except NotImplementedError as e:
        print(f'Error loading MAT file with loadmat: {e}')
        print('Attempting to load using h5py...')
        try:
            with h5py.File(file_path, 'r') as f:
                train_dataX = np.array(f['train_dataX'][:])
                train_dataY = np.array(f['train_dataY'][:])
                test_dataX = np.array(f['test_dataX'][:])
                test_dataY = np.array(f['test_dataY'][:])
        except Exception as e:
            print(f'Error loading MAT file with h5py: {e}')
            raise
    except Exception as e:
        print(f'Unexpected error: {e}')
        raise
def load_feature_means(file_path):
    feature_means = loadmat(file_path)['features_means']
    feature_means = np.expand_dims(feature_means, axis=0)
    return feature_means
def create_masked_data(x_train, feature_means, mask_probability=0.1):
    mask = np.random.rand(*x_train.shape) < mask_probability
    # Reshape feature_means to be broadcastable
    feature_means_reshaped = feature_means.reshape(1, 1, -1)
    x_train_masked = np.where(mask, feature_means_reshaped, x_train)
    return x_train_masked
def save_reconstructions(file_path, reconstructions):
    savemat(file_path, reconstructions)
def get_predictions(data,weights_path):
    autoenc = Autoencoder(data.shape[1:])
    autoenc.model.load_weights(weights_path)
    return autoenc.predict(data)
def trainmodel(traindatapath, feature_means_path, model_save_path, v73, model_params):
    try:
        # Load data
        if v73:
            x_train = load_v73_mat_file(traindatapath, var_name='train_dataX')
            y_train = load_v73_mat_file(traindatapath, var_name='train_dataY')
            x_test = load_v73_mat_file(traindatapath, var_name='test_dataX')
            y_test = load_v73_mat_file(traindatapath, var_name='test_dataY')
        else:
            data = loadmat(traindatapath)
            x_train, y_train = data['train_dataX'], data['train_dataY']
            x_test, y_test = data['test_dataX'], data['test_dataY']
        print(f"Data loaded. Shapes: x_train: {x_train.shape}, y_train: {y_train.shape}")
        # Load feature means and create masked data
        feature_means = load_feature_means(feature_means_path)
        x_train_masked = create_masked_data(x_train, feature_means)
        # Prepare validation data
        validation_split = float(model_params['val_split'])  # Ensure this is a float
        val_size = int(len(x_train) * validation_split)
        x_val, y_val = x_train[:val_size], y_train[:val_size]
        x_val_masked = x_train_masked[:val_size]
        x_train, y_train = x_train[val_size:], y_train[val_size:]
        x_train_masked = x_train_masked[val_size:]
        print(f"Data prepared. Shapes: x_train_masked: {x_train_masked.shape}, x_val_masked: {x_val_masked.shape}")
        # Ensure input_shape is a tuple of integers
        input_shape = tuple(map(int, model_params['input_shape']))
        print(f"Creating Autoencoder with input_shape: {input_shape}")
        # Create and train the autoencoder
        autoenc = Autoencoder(
            input_shape=input_shape,
            bottleneck_size=int(model_params['bottleneck_size']),
            activation=model_params['activation'],
            conv_units=int(model_params['conv_units']),
            kernel_size=int(model_params['kernel_size']),
            stride_size=int(model_params['stride_size']),
            batch_size=int(model_params['batch_size']),
            dropout_rate=float(model_params['dropout_rate'])
        )
        print("Autoencoder created. Starting training...")
        history = autoenc.train(
            x_train_masked, y_train,
            x_val_masked, y_val,
            epochs=int(model_params['epochs']),
            ER_Patience=int(model_params['ER_Patience']),
            LR_patience=int(model_params['LR_patience'])
        )
        print("Training completed. Evaluating model...")
        # Evaluate and save the model
        test_loss = autoenc.evaluate(x_test, y_test)
        print(f'Test loss: {test_loss}')
        autoenc.save_model(model_save_path)
        return test_loss
    except Exception as e:
        print(f"An error occurred: {str(e)}")
        import traceback
        traceback.print_exc()
        return None
if __name__ == "__main__":
    # Load your data
    input_shape = [15, 44]
    data = np.random.rand(1000,15,44)
    # get_predictions(data)
    autoenc = Autoencoder(input_shape)
    model = autoenc.build_model()
    model.summary()
    file_path = '/Users/maxweinberg/Desktop/Bergan_Lab_Repo/All_Fiber_Stuff/Sleapproc/Autoenc/PreprocForEncoder/traintestSeq.mat'
    x_train = load_v73_mat_file(file_path, var_name='train_dataX')
    y_train = load_v73_mat_file(file_path, var_name='train_dataY')
    x_test = load_v73_mat_file(file_path, var_name='test_dataX')
    y_test = load_v73_mat_file(file_path, var_name='test_dataY')
    print(x_train.shape)
    feature_means = load_feature_means('/Users/maxweinberg/Desktop/Bergan_Lab_Repo/All_Fiber_Stuff/Sleapproc/Autoenc/PreprocForEncoder/features_means.mat')
    # Create masked data
    x_train_masked = create_masked_data(x_train, feature_means)
    # Define the shape of your input data
    input_shape = x_train.shape[1:]
    print(input_shape)
    # Manually split the data into training and validation sets to avoid temporal leakage
    validation_split = 0.2
    val_size = int(len(x_train) * validation_split)
    x_val = x_train[:val_size]
    y_val = y_train[:val_size]
    x_val_masked = x_train_masked[:val_size]
    x_train = x_train[val_size:]
    y_train = y_train[val_size:]
    x_train_masked = x_train_masked[val_size:]
    # # # Define the parameter space for Bayesian optimization
    # # param_space = [
    # #     Integer(22, 660, name='bottleneck_size')
    # # ]
    # # # Perform Bayesian search
    # # n_calls = 20  # Number of parameter settings that are sampled
    # # res = bayesian_search(input_shape, param_space, x_train_masked, y_train, x_val_masked, y_val, x_test, y_test, n_calls=n_calls)
    # # best_params = res.x
    # # best_test_loss = res.fun
    # # print(f"Best parameters: {best_params}")
    # # print(f"Best test loss: {best_test_loss}")
    # # savemat('gridresults.mat', {'best_params': best_params, 'best_test_loss': best_test_loss})
    # Initialize and train the best autoencoder
    autoenc = Autoencoder(
        input_shape
    )
    autoenc.train(x_train_masked, y_train, x_val_masked, y_val)
    # # Get the reconstructions
    # reconstructions = autoenc.predict(x_test)
    # sample_data = loadmat('/Users/maxweinberg/Desktop/Bergan_Lab_Repo/All_Fiber_Stuff/Sleapproc/Autoenc/PreprocForEncoder/sequences_fp.mat')['sequences_fp']
    # reconstructions2 = autoenc.predict(sample_data)
    # # Save the reconstructions to a .mat file
    # save_reconstructions('/Users/maxweinberg/Desktop/Bergan_Lab_Repo/All_Fiber_Stuff/Sleapproc/Autoenc/PreprocForEncoder/reconstructions.mat', {'reconstructions': reconstructions})
    # save_reconstructions('/Users/maxweinberg/Desktop/Bergan_Lab_Repo/All_Fiber_Stuff/Sleapproc/Autoenc/PreprocForEncoder/reconstructions2.mat', {'reconstructions2': reconstructions2})
    # # Evaluate the model on the test data
    test_loss = autoenc.evaluate(x_test, y_test)
    print(f'Test loss: {test_loss}')
    # # Save the entire model to a file
    model_save_path = '/Users/maxweinberg/Desktop/Bergan_Lab_Repo/All_Fiber_Stuff/Sleapproc/Autoenc/Encoder/models/conv_autoencoder_model.h5'
    autoenc.save_model(model_save_path)

## another autoencoder

### load data

In [7]:
def load_video(video_path, target_size=None):
    """
    Load an MP4 video into a numpy array (grayscale).
    
    Args:
        video_path: Path to the MP4 file.
        target_size: (width, height) to resize frames (optional).
    
    Returns:
        video_array: Shape (num_frames, height, width)
    """
    cap = cv2.VideoCapture(video_path)
    frames = []
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        if target_size:
            gray = cv2.resize(gray, target_size)  # (width, height)
        frames.append(gray)
    
    cap.release()
    video_array = np.array(frames)  # Shape: (num_frames, height, width)
    return video_array

# Usage
video_path = "/home/mingxiao/Desktop/jellyfish/video/video_1_clips/c1_high_res_10min_track_reencoded.mp4"
video_data = load_video(video_path)
print(video_data.shape)  # Should output (90003, 170, 174)

(90003, 170, 174)


In [8]:
all_tracked_points = np.load('/home/mingxiao/Desktop/jellyfish/video/video_1_clips/all_points_corrected_10min.npy')
print(all_tracked_points.shape)

(90002, 17, 2)


In [9]:
if video_data.shape[0] > all_tracked_points.shape[0]:
    video_data = video_data[video_data.shape[0] - all_tracked_points.shape[0]:, :, :]
print(video_data.shape)
print(all_tracked_points.shape)

(90002, 170, 174)
(90002, 17, 2)


#### convert numpy data to tensor

In [14]:
batch_size

64

In [13]:
# Create a dataset from your video data
dataset = tf.data.Dataset.from_tensor_slices(video_data)
dataset = dataset.map(lambda x: tf.cast(x, tf.float32) / 255.0)
dataset = dataset.map(lambda x: tf.expand_dims(x, axis=-1))
dataset = dataset.batch(batch_size)

In [17]:
batch_size = 64  # Adjust based on your GPU memory
num_batches = len(video_data) // batch_size

processed_batches = []
for i in range(num_batches):
    start_idx = i * batch_size
    end_idx = (i + 1) * batch_size
    
    # Process each batch with float16
    batch = video_data[start_idx:end_idx]
    batch_tensor = tf.cast(batch, tf.float16) / 255.0
    batch_tensor = tf.expand_dims(batch_tensor, axis=-1)
    processed_batches.append(batch_tensor)

# Combine all processed batches
video_tensor = tf.concat(processed_batches, axis=0)

2025-02-02 17:10:09.759504: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:497] Allocator (GPU_0_bfc) ran out of memory trying to allocate 3.61MiB (rounded to 3786240)requested by op RealDiv
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2025-02-02 17:10:09.759566: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1053] BFCAllocator dump for GPU_0_bfc
2025-02-02 17:10:09.759590: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1060] Bin (256): 	Total Chunks: 33, Chunks in use: 33. 8.2KiB allocated for chunks. 8.2KiB in use in bin. 2.3KiB client-requested in use in bin.
2025-02-02 17:10:09.759609: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1060] Bin (512): 	Total Chunks: 1, Chunks in use: 1. 512B allocated for chunks. 512B in use in bin. 512B client-requested in use in bin.
2025-02-02 17:10:09.75

ResourceExhaustedError: {{function_node __wrapped__RealDiv_device_/job:localhost/replica:0/task:0/device:GPU:0}} failed to allocate memory [Op:RealDiv] name: 

In [15]:
video_tensor = tf.constant(video_data, dtype=tf.float16) / 255.0
video_tensor = tf.expand_dims(video_tensor, axis=-1)  # Add channel dim
print(video_tensor.shape)

2025-02-02 17:09:12.658772: E tensorflow/core/util/util.cc:131] oneDNN supports DT_HALF only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.
2025-02-02 17:09:24.262774: W external/local_xla/xla/tsl/framework/bfc_allocator.cc:497] Allocator (GPU_0_bfc) ran out of memory trying to allocate 4.96GiB (rounded to 5324518400)requested by op RealDiv
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2025-02-02 17:09:24.262843: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1053] BFCAllocator dump for GPU_0_bfc
2025-02-02 17:09:24.262868: I external/local_xla/xla/tsl/framework/bfc_allocator.cc:1060] Bin (256): 	Total Chunks: 33, Chunks in use: 33. 8.2KiB allocated for chunks. 8.2KiB in use in bin. 2.3KiB client-requested in use in bin.
2025-02-02 17:09:24.262888: I external/local_xla/xla/

ResourceExhaustedError: {{function_node __wrapped__RealDiv_device_/job:localhost/replica:0/task:0/device:GPU:0}} failed to allocate memory [Op:RealDiv] name: 

In [13]:
# Assuming `all_tracked_points` is a numpy array of shape (90003, 17, 2)
frame_cnt, x, y = video_tensor.shape[:3]
points_tensor = tf.constant(all_tracked_points, dtype=tf.float32)
canvas_size = tf.constant([y, x], dtype=tf.float32)  # (width, height)
points_tensor = points_tensor / canvas_size
print(points_tensor.shape)

(90002, 17, 2)


#### add noise to points

In [14]:
# Add Gaussian noise and shuffle identities
def create_noisy_points(points_tensor, noise_std=0.05):
    noise = tf.random.normal(tf.shape(points_tensor), stddev=noise_std)
    noisy_points = points_tensor + noise
    
    # Shuffle point identities (axis=1)
    shuffled_points = tf.random.shuffle(tf.transpose(noisy_points, perm=[0, 2, 1]))
    return tf.transpose(shuffled_points, perm=[0, 2, 1])

noisy_points_tensor = create_noisy_points(points_tensor)

### construct model

In [3]:
def build_gpu_model():
    # Inputs
    video_input = layers.Input(shape=(170, 174, 1), name="video_input")
    points_input = layers.Input(shape=(17, 2), name="points_input")
    
    # Video encoder
    x = layers.Conv2D(32, (3, 3), activation="relu", padding="same")(video_input)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Conv2D(64, (3, 3), activation="relu", padding="same")(x)
    x = layers.MaxPooling2D((2, 2))(x)
    video_features = x
    video_tokens = layers.Reshape((-1, 64))(video_features)
    
    # Point encoder
    point_queries = layers.Dense(64, activation="relu")(points_input)
    
    # Cross-attention (GPU-optimized implementation)
    attention = layers.MultiHeadAttention(num_heads=4, key_dim=16)(point_queries, video_tokens)
    x = layers.Concatenate(axis=-1)([attention, point_queries])
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dense(32, activation="relu")(x)
    corrected_points = layers.Dense(2, name="corrected_points")(x)
    
    model = tf.keras.Model(
        inputs=[video_input, points_input],
        outputs=corrected_points
    )
    return model

model = build_gpu_model()
model.summary()

I0000 00:00:1738544576.311952  347075 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 18429 MB memory:  -> device: 0, name: NVIDIA RTX A4500, pci bus id: 0000:17:00.0, compute capability: 8.6
I0000 00:00:1738544576.312882  347075 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 17833 MB memory:  -> device: 1, name: NVIDIA RTX A4500, pci bus id: 0000:73:00.0, compute capability: 8.6


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ video_input         │ (None, 170, 174,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 170, 174,  │        320 │ video_input[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 85, 87,    │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 85, 87,    │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ points_input        │ (None, 17, 2)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 42, 43,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 17, 64)    │        192 │ points_input[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 1806, 64)  │          0 │ max_pooling2d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 17, 64)    │     16,640 │ dense[0][0],      │
│ (MultiHeadAttentio… │                   │            │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 17, 128)   │          0 │ multi_head_atten… │
│ (Concatenate)       │                   │            │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 17, 64)    │      8,256 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 17, 32)    │      2,080 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ corrected_points    │ (None, 17, 2)     │         66 │ dense_2[0][0]     │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 46,050 (179.88 KB)

 Trainable params: 46,050 (179.88 KB)

 Non-trainable params: 0 (0.00 B)

### training

In [4]:
batch_size = 64

# Create dataset from tensors
dataset = tf.data.Dataset.from_tensor_slices(
    ((video_tensor, noisy_points_tensor), points_tensor)
).batch(batch_size).prefetch(tf.data.AUTOTUNE)

NameError: name 'video_tensor' is not defined

In [5]:
type(dataset)

NameError: name 'dataset' is not defined

In [6]:
# Calculate split index (e.g., 90% train, 10% validation)
split_idx = int(0.9 * len(video_tensor))  # Assuming video_tensor has 90003 frames

# Training data
train_video = video_tensor[:split_idx]
train_noisy = noisy_points_tensor[:split_idx]
train_true = points_tensor[:split_idx]

# Validation data
val_video = video_tensor[split_idx:]
val_noisy = noisy_points_tensor[split_idx:]
val_true = points_tensor[split_idx:]

NameError: name 'video_tensor' is not defined

In [29]:
batch_size = 64

# Training dataset
train_dataset = tf.data.Dataset.from_tensor_slices(
    ((train_video, train_noisy), train_true)
).batch(batch_size).prefetch(tf.data.AUTOTUNE)

# Validation dataset
val_dataset = tf.data.Dataset.from_tensor_slices(
    ((val_video, val_noisy), val_true)
).batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [30]:
history = model.fit(
    train_dataset,
    epochs=20,
    validation_data=val_dataset,  # Use explicit validation dataset
    verbose=1
)

Epoch 1/20


1266/1266 ━━━━━━━━━━━━━━━━━━━━ 152s 113ms/step - loss: 0.0360 - val_loss: 0.0248
Epoch 2/20
1266/1266 ━━━━━━━━━━━━━━━━━━━━ 143s 113ms/step - loss: 0.0301 - val_loss: 0.0215
Epoch 3/20
 908/1266 ━━━━━━━━━━━━━━━━━━━━ 38s 109ms/step - loss: 0.0318

KeyboardInterrupt: 